In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Conv1D, GlobalAveragePooling1D, Bidirectional, Dropout
from transformers import BertTokenizer, TFBertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.layers import Reshape

from tensorflow.keras.layers import Dense, Dropout, Input, Layer
from tensorflow.keras.utils import register_keras_serializable
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from lime.lime_text import LimeTextExplainer

C:\Users\isaac\anaconda3\envs\py310\lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


In [4]:
# Combine fake and real news into a single dataframe
fake_news = pd.read_csv('fake_cleaned.csv')
real_news = pd.read_csv('true_cleaned.csv')

fake_news['label'] = 0  # 0 for fake
real_news['label'] = 1  # 1 for real

min_samples = min(len(real_news), len(fake_news))

data = pd.concat([
    real_news.sample(min_samples, random_state=42),
    fake_news.sample(min_samples, random_state=42)
]).sample(frac=1, random_state=42)  # Shuffle

# Prepare dataset for BERT
tokenizer_bert = BertTokenizer.from_pretrained('bert-base-uncased')
def encode_texts(texts):
    return tokenizer_bert(texts.tolist(), padding=True, truncation=True, max_length=256, return_tensors='np')


# Evaluate models
def evaluate_model(model, X_test, y_test, bert=False):
    if bert:
        preds = (model.predict((X_test['input_ids'], X_test['attention_mask'])) > 0.5).astype('int32')
    else:
        preds = (model.predict(X_test) > 0.5).astype('int32')
    print(classification_report(y_test, preds))

In [5]:
# Prepare tokenized inputs
X = encode_texts(data["text"])
y = np.array(data["label"])  # Labels (0 = Fake, 1 = Real)


train_idx, test_idx = train_test_split(
    np.arange(len(y)), 
    test_size=0.2, 
    stratify=y, 
    random_state=42
)

# Split X and y using the indices
X_train = {k: v[train_idx] for k, v in X.items()}
X_test = {k: v[test_idx] for k, v in X.items()}
y_train, y_test = y[train_idx], y[test_idx]

In [16]:
from transformers import TFBertForSequenceClassification, BertTokenizer, create_optimizer

optimizer, _ = create_optimizer(init_lr=5e-5, num_train_steps=1000, num_warmup_steps=100)

bert_classifier = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=1  # Binary classification
)


input_ids = Input(shape=(256,), dtype=tf.int32, name="input_ids")
attention_mask = Input(shape=(256,), dtype=tf.int32, name="attention_mask")


logits = bert_classifier(input_ids, attention_mask=attention_mask).logits
probabilities = tf.sigmoid(logits)

bert_classifier = Model(inputs=[input_ids, attention_mask], outputs=probabilities)
optimizer = tf.keras.optimizers.Adam(learning_rate=3e-5)

bert_classifier.compile(optimizer=optimizer, 
                        loss=tf.keras.losses.BinaryCrossentropy(from_logits=False), 
                        metrics=["accuracy"])



All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
import numpy as np
print(f"Class distribution: {np.unique(y_train, return_counts=True)}")

Class distribution: (array([0, 1], dtype=int64), array([16968, 16968], dtype=int64))


In [20]:
from tensorflow.keras.callbacks import EarlyStopping

# Early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train model
bert_classifier.fit(
    x={"input_ids": X_train["input_ids"], "attention_mask": X_train["attention_mask"]},
    y=y_train,
    validation_data=({"input_ids": X_test["input_ids"], "attention_mask": X_test["attention_mask"]}, y_test),
    epochs=10,
    batch_size=8,
    callbacks=[early_stopping]
)


Epoch 1/10
4242/4242 [==============================] - 467s 109ms/step - loss: 0.0136 - accuracy: 0.9964 - val_loss: 0.0011 - val_accuracy: 0.9999
Epoch 2/10
4242/4242 [==============================] - 460s 108ms/step - loss: 0.0344 - accuracy: 0.9879 - val_loss: 0.0012 - val_accuracy: 0.9999
Epoch 3/10
4242/4242 [==============================] - 460s 108ms/step - loss: 0.0075 - accuracy: 0.9989 - val_loss: 0.0059 - val_accuracy: 0.9993
Epoch 4/10
4242/4242 [==============================] - 460s 108ms/step - loss: 0.2316 - accuracy: 0.8847 - val_loss: 0.0071 - val_accuracy: 0.9992


In [21]:
sample_text = ["Head of a conservative Republican faction in the U.S. Congress urged budget restraint in 2019"]

# Tokenize the input text
tokenized_input = tokenizer_bert(
    sample_text, 
    padding="max_length", 
    truncation=True, 
    max_length=256, 
    return_tensors="tf"
)

# Convert tokenized input to the required format
model_input = {
    "input_ids": tf.convert_to_tensor(tokenized_input["input_ids"]),
    "attention_mask": tf.convert_to_tensor(tokenized_input["attention_mask"])
}


In [22]:
# Get raw logits from the model
logits = bert_classifier.predict(model_input)

# Convert logits to probability using sigmoid (since it's binary classification)
probabilities = tf.nn.sigmoid(logits).numpy()[0]

# Define a threshold (default is 0.5)
threshold = 0.5  
predicted_label = int(probabilities[0] > threshold)  # Convert probability to class

# Map Predictions to Labels (Modify if needed)
label_map = {0: "Fake News", 1: "Real News"}

# Print results
print(f"Prediction: {label_map[predicted_label]}")
print(f"Probability Score: {probabilities[0]:.4f}")  # Print probability score
print(f"Confidence: {probabilities[0] * 100:.2f}%")  # Print confidence in prediction


1/1 [==============================] - 2s 2s/step
Prediction: Real News
Probability Score: 0.5014
Confidence: 50.14%


In [23]:
unique, counts = np.unique(y_train, return_counts=True)
print(dict(zip(unique, counts)))


{0: 16968, 1: 16968}


In [24]:
logits = bert_classifier.predict(model_input)
probabilities = tf.nn.sigmoid(logits).numpy()[0]  # Convert logits to probability

print(f"Raw Logits: {logits}")
print(f"Probability Score: {probabilities[0]:.4f}")

threshold = 0.5
predicted_label = int(probabilities[0] > threshold)
print(f"Final Prediction: {predicted_label}")


1/1 [==============================] - 0s 99ms/step
Raw Logits: [[0.00562548]]
Probability Score: 0.5014
Final Prediction: 1


In [26]:
# Save model
bert_classifier.save_weights('bert_redo.weights.h5')
bert_classifier.save("bert_redo.keras")
bert_classifier.save("bert_redo.h5")
bert_classifier.save("bert_redo_99")

INFO:tensorflow:Assets written to: bert_redo_99\assets


INFO:tensorflow:Assets written to: bert_redo_99\assets
